In [6]:
import os
import requests
import pandas as pd

from io import BytesIO
from pathlib import Path
from datetime import datetime
from zoneinfo import ZoneInfo
from dotenv import load_dotenv
from azure.storage.filedatalake import DataLakeServiceClient


# 1. DEFINIR DATA ATUAL

agora = datetime.now(ZoneInfo("America/Sao_Paulo"))
hoje = agora.strftime("%m-%d-%Y")


# 2. DEFINIR API BACEN

url = (
    "https://olinda.bcb.gov.br/olinda/servico/PTAX/"
    "versao/v1/odata/"
    "CotacaoDolarDia(dataCotacao=@dataCotacao)"
)

parametros = {
    "@dataCotacao": f"'{hoje}'",
    "$format": "json"
}


# 3. CONSULTAR API

resposta = requests.get(
    url,
    params=parametros,
    timeout=30
)

resposta.raise_for_status()

df = pd.DataFrame(resposta.json()["value"])


# 4. VERIFICAR E SELECIONAR COTAÇÃO

if df.empty:

    print("Nenhuma cotação disponível para hoje.")

else:

    df["dataHoraCotacao"] = pd.to_datetime(
        df["dataHoraCotacao"]
    )

    df = df.sort_values(
        "dataHoraCotacao"
    ).tail(1).copy()

    df = df[
        [
            "dataHoraCotacao",
            "cotacaoCompra",
            "cotacaoVenda"
        ]
    ]


    # 5. CONECTAR AO AZURE

    load_dotenv()

    connection_string = os.getenv(
        "AZURE_STORAGE_CONNECTION_STRING"
    )

    if not connection_string:
        raise ValueError("Credencial Azure não encontrada.")

    datalake_service = DataLakeServiceClient.from_connection_string(
        connection_string
    )

    file_system = datalake_service.get_file_system_client(
        file_system="bronze"
    )


    # 6. DEFINIR ARQUIVO HISTÓRICO NA BRONZE

    caminho = (
        "api/bacen/"
        "cotacao_dolar_historico.parquet"
    )

    arquivo_azure = file_system.get_file_client(caminho)


    # 7. LER HISTÓRICO DA BRONZE

    dados_historico = arquivo_azure.download_file().readall()

    historico = pd.read_parquet(
        BytesIO(dados_historico)
    )

    historico["dataHoraCotacao"] = pd.to_datetime(
        historico["dataHoraCotacao"]
    )


    # 8. ACRESCENTAR COTAÇÃO DO DIA

    historico = pd.concat(
        [historico, df],
        ignore_index=True
    )


    # 9. ELIMINAR DUPLICIDADES POR DATA

    historico["data"] = historico["dataHoraCotacao"].dt.date

    historico = historico.sort_values(
        "dataHoraCotacao"
    )

    historico = historico.drop_duplicates(
        subset=["data"],
        keep="last"
    )

    historico = historico.drop(columns=["data"])


    # 10. ATUALIZAR ARQUIVO LOCAL CSV

    arquivo_local = Path(
        "data/arquivos_api/cotacao_dolar_historico.csv"
    )

    arquivo_local.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    historico.to_csv(
        arquivo_local,
        index=False,
        encoding="utf-8-sig"
    )

    print("CSV local atualizado:", arquivo_local)


    # 11. CONVERTER HISTÓRICO PARA PARQUET

    buffer = BytesIO()

    historico.to_parquet(
        buffer,
        index=False,
        engine="pyarrow"
    )


    # 12. ATUALIZAR ARQUIVO NA BRONZE

    arquivo_azure.upload_data(
        buffer.getvalue(),
        overwrite=True
    )

    print("Histórico atualizado na Bronze!")
    print("Total de registros:", len(historico))
    print("Arquivo Azure:", caminho)


CSV local atualizado: data\arquivos_api\cotacao_dolar_historico.csv
Histórico atualizado na Bronze!
Total de registros: 452
Arquivo Azure: api/bacen/cotacao_dolar_historico.parquet
